## Projects



In [1]:
import cv2
import numpy as np
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report

from tensorflow.keras.utils import to_categorical
from skimage.feature import hog
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

from tensorflow.keras.utils import to_categorical

In [2]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

In [3]:
dataset_path = "dataset"

images = []
labels = []

label_map = {}
current_label = 0

IMG_SIZE = 128

In [4]:
for person_name in os.listdir(dataset_path):

    person_folder = os.path.join(
        dataset_path,
        person_name
    )

    if not os.path.isdir(person_folder):
        continue

    label_map[current_label] = person_name

    for image_name in os.listdir(person_folder):

        image_path = os.path.join(
            person_folder,
            image_name
        )

        image = cv2.imread(image_path)

        if image is None:
            continue

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5
        )

        for (x, y, w, h) in faces:

            face = gray[y:y+h, x:x+w]

            face = cv2.resize(
                face,
                (IMG_SIZE, IMG_SIZE)
            )

            images.append(face)
            labels.append(current_label)

    current_label += 1

In [5]:
X = np.array(images)
y = np.array(labels)

print("Images:", X.shape)
print("Labels:", y.shape)

Images: (12, 128, 128)
Labels: (12,)


In [6]:
X = X / 255.0

In [7]:
hog_features = []

for image in X:

    feature = hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )

    hog_features.append(feature)

hog_features = np.array(hog_features)

print(hog_features.shape)

(12, 8100)


In [8]:
X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(
    hog_features,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
svm_model = SVC(
    kernel='linear',
    probability=True
)

svm_model.fit(
    X_train_ml,
    y_train_ml
)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [10]:
svm_predictions = svm_model.predict(X_test_ml)

print(
    classification_report(
        y_test_ml,
        svm_predictions
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



In [11]:
X_dl = np.expand_dims(X, axis=-1)

y_dl = to_categorical(y)

X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_dl,
    y_dl,
    test_size=0.2,
    random_state=42
)

In [12]:
cnn_model = Sequential([

    Conv2D(
        32,
        (3, 3),
        activation='relu',
        input_shape=(128, 128, 1)
    ),

    MaxPooling2D(2, 2),

    Conv2D(
        64,
        (3, 3),
        activation='relu'
    ),

    MaxPooling2D(2, 2),

    Conv2D(
        128,
        (3, 3),
        activation='relu'
    ),

    MaxPooling2D(2, 2),

    Flatten(),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.5),

    Dense(
        len(label_map),
        activation='softmax'
    )
])

In [13]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [14]:
cnn_model.fit(
    X_train_dl,
    y_train_dl,
    validation_data=(
        X_test_dl,
        y_test_dl
    ),
    epochs=10,
    batch_size=32
)

Epoch 1/10


1/1 [==============================] - 1s 629ms/step - loss: 0.6928 - accuracy: 0.3333 - val_loss: 0.5390 - val_accuracy: 1.0000
Epoch 2/10
1/1 [==============================] - 0s 37ms/step - loss: 0.6211 - accuracy: 0.6667 - val_loss: 0.4391 - val_accuracy: 1.0000
Epoch 3/10
1/1 [==============================] - 0s 38ms/step - loss: 0.6209 - accuracy: 0.7778 - val_loss: 0.2227 - val_accuracy: 1.0000
Epoch 4/10
1/1 [==============================] - 0s 36ms/step - loss: 0.3275 - accuracy: 0.8889 - val_loss: 0.1314 - val_accuracy: 1.0000
Epoch 5/10
1/1 [==============================] - 0s 37ms/step - loss: 0.0909 - accuracy: 1.0000 - val_loss: 0.1344 - val_accuracy: 1.0000
Epoch 6/10
1/1 [==============================] - 0s 36ms/step - loss: 0.0868 - accuracy: 1.0000 - val_loss: 0.0228 - val_accuracy: 1.0000
Epoch 7/10
1/1 [==============================] - 0s 37ms/step - loss: 0.0592 - accuracy: 1.0000 - val_loss: 0.0025 - val_accuracy: 1.0000
Epoch 8/10
1/1 [========

In [15]:
cnn_model.save("face_model.keras")

In [9]:

model = load_model("face_model.keras")

IMG_SIZE = 128

label_map = {
    0: "person1",
    1: "person2"
}

test_folder = "test_images"

for img_name in os.listdir(test_folder):

    img_path = os.path.join(test_folder, img_name)
    frame = cv2.imread(img_path)

    if frame is None:
        continue

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    face = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))

    face = face / 255.0
    face = np.expand_dims(face, axis=-1)
    face = np.expand_dims(face, axis=0)

    pred = model.predict(face, verbose=0)

    class_id = np.argmax(pred)
    name = label_map.get(class_id, "Unknown")
    conf = np.max(pred)

    print(f"{img_name} -> {name} ({conf:.2f})")

download (10).jpg -> person1 (0.67)


## DataSets

In [9]:
import cv2
import os
import numpy as np

def augment_image(img):

    h, w = img.shape

    augmented = []

    # 1. original
    augmented.append(img)

    # 2. flip
    augmented.append(cv2.flip(img, 1))

    # 3. blur
    augmented.append(cv2.GaussianBlur(img, (5,5), 0))

    # 4. brightness
    bright = cv2.convertScaleAbs(img, alpha=1.2, beta=20)
    augmented.append(bright)

    # 5. noise
    noise = img + np.random.normal(0, 10, img.shape)
    noise = np.clip(noise, 0, 255).astype(np.uint8)
    augmented.append(noise)

    return augmented

In [10]:
input_path = "dataset"
output_path = "dataset_aug"

IMG_SIZE = 128

os.makedirs(output_path, exist_ok=True)

for person in os.listdir(input_path):

    person_in = os.path.join(input_path, person)
    person_out = os.path.join(output_path, person)

    os.makedirs(person_out, exist_ok=True)

    count = 0

    for img_name in os.listdir(person_in):

        img_path = os.path.join(person_in, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        aug_list = augment_image(img)

        for aug in aug_list:

            save_path = os.path.join(
                person_out,
                f"{count}.jpg"
            )

            cv2.imwrite(save_path, aug)
            count += 1

    print(person, "generated:", count)

person1 generated: 5
person2 generated: 5
